### CEO Lakebase (Autoscaling)

Provisions a **Lakebase Autoscaling** project (`w.postgres`) for the CEO Dashboard app
and creates the session/message tables.

The project creator is automatically granted superuser in the `databricks_postgres` database,
so DDL runs cleanly from this stage notebook.

In [ ]:
%pip install --upgrade "databricks-sdk>=0.81.0" "psycopg[binary]>=3.0"

In [ ]:
dbutils.library.restartPython()

In [ ]:
import re
CATALOG = dbutils.widgets.get("CATALOG")
PROJECT_ID = re.sub(r'[^a-z0-9-]', '-', f"{CATALOG}-ceo-sessions".lower())
ENDPOINT_PATH = f"projects/{PROJECT_ID}/branches/production/endpoints/primary"

In [ ]:
# packages installed in cell 1

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.postgres import Project, ProjectSpec
import time, sys

sys.path.append('../utils')
from uc_state import add

w = WorkspaceClient()

# Create or reuse Autoscale project
is_new = False
try:
    project = w.postgres.get_project(name=f"projects/{PROJECT_ID}")
    print(f"\u267b\ufe0f Reusing existing Lakebase project: {PROJECT_ID}")
except Exception:
    is_new = True
    print(f"Creating Lakebase Autoscale project: {PROJECT_ID}")
    op = w.postgres.create_project(
        project=Project(spec=ProjectSpec(display_name=f"CEO Sessions ({CATALOG})", pg_version="17")),
        project_id=PROJECT_ID,
    )
    project = op.result()
    add(CATALOG, "postgres_projects", {"project_id": PROJECT_ID, "name": project.name})
    print(f"\u2705 Created project: {project.name}")

# For existing projects the endpoint is already RUNNING — only poll when freshly created
if is_new:
    print(f"Waiting for endpoint to be ready...")
    for attempt in range(40):
        try:
            ep = w.postgres.get_endpoint(name=ENDPOINT_PATH)
            state = str(ep.status.current_state) if ep.status else "UNKNOWN"
            if "RUNNING" in state or "AVAILABLE" in state or "ACTIVE" in state:
                print(f"\u2705 Endpoint ready (state={state})")
                break
            print(f"  [{attempt+1}/40] state={state}, retrying in 15s...")
        except Exception as e:
            print(f"  [{attempt+1}/40] not yet reachable: {e}, retrying in 15s...")
        time.sleep(15)
    else:
        raise TimeoutError(f"Endpoint {ENDPOINT_PATH} did not start within 10 minutes")
else:
    ep = w.postgres.get_endpoint(name=ENDPOINT_PATH)
    print(f"\u2705 Endpoint ready (state={ep.status.current_state})")

##### Connect and create session tables

In [ ]:
import psycopg

ep = w.postgres.get_endpoint(name=ENDPOINT_PATH)
host = ep.status.hosts.host
user = w.current_user.me().user_name

cred = w.postgres.generate_database_credential(endpoint=ENDPOINT_PATH)

conn_str = f"host={host} dbname=databricks_postgres user={user} password={cred.token} sslmode=require"

# Project creator has superuser — CREATE TABLE works without any GRANT
STATEMENTS = [
    "CREATE TABLE IF NOT EXISTS ceo_sessions (id UUID PRIMARY KEY DEFAULT gen_random_uuid(), title TEXT NOT NULL DEFAULT 'New Session', created_at TIMESTAMPTZ NOT NULL DEFAULT NOW(), updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW())",
    "CREATE TABLE IF NOT EXISTS ceo_messages (id UUID PRIMARY KEY DEFAULT gen_random_uuid(), session_id UUID NOT NULL REFERENCES ceo_sessions(id) ON DELETE CASCADE, role TEXT NOT NULL CHECK (role IN ('user', 'assistant')), content TEXT NOT NULL, created_at TIMESTAMPTZ NOT NULL DEFAULT NOW(), documents_referenced JSONB DEFAULT '[]'::jsonb)",
    "CREATE INDEX IF NOT EXISTS idx_ceo_messages_session ON ceo_messages(session_id, created_at)",
    "CREATE INDEX IF NOT EXISTS idx_ceo_sessions_updated ON ceo_sessions(updated_at DESC)",
]

with psycopg.connect(conn_str) as conn:
    with conn.cursor() as cur:
        for stmt in STATEMENTS:
            cur.execute(stmt)
    conn.commit()

print(f"\u2705 Tables ready in {PROJECT_ID}/databricks_postgres")
print(f"   Host: {host}")

In [ ]:
print(f"\u2705 CEO Lakebase stage complete")
print(f"   Project: {PROJECT_ID}")
print(f"   Endpoint: {ENDPOINT_PATH}")
print(f"   Host: {host}")